In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

teacher_path = "/kaggle/input/datasets/rishwanthj/densenet121/best_densenet121.pth"

teacher_model = models.densenet121(weights=None)
teacher_model.classifier = nn.Linear(1024, num_classes)

teacher_model.load_state_dict(torch.load(teacher_path, map_location=device))
teacher_model = teacher_model.to(device)
teacher_model.eval()

for p in teacher_model.parameters():
    p.requires_grad = False

In [ ]:
from torchvision import transforms

image_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),   # Unified input size
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],   # ImageNet mean
            std=[0.229, 0.224, 0.225]     # ImageNet std
        )
    ]),
    
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}

In [ ]:
data_dir = "/kaggle/input/xray-ds/split_dataset"

train_dataset = datasets.ImageFolder(
    root=f"{data_dir}/train",
    transform=image_transforms['train']
)
val_dataset = datasets.ImageFolder(
    root=f"{data_dir}/val",
    transform=image_transforms['val']
)
test_dataset = datasets.ImageFolder(
    root=f"{data_dir}/test",
    transform=image_transforms['test']
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
def get_class_counts(dataset):
    """
    Computes number of samples per class from ImageFolder dataset.
    """
    targets = dataset.targets
    class_to_idx = dataset.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    counts = {}
    for idx in idx_to_class:
        counts[idx] = targets.count(idx)

    return counts


# Compute counts from TRAIN dataset only
class_counts = get_class_counts(train_dataset)

classes = list(class_counts.keys())
counts = list(class_counts.values())

total_samples = sum(counts)
num_classes = len(classes)

weights = []
for i in range(num_classes):
    weight = total_samples / (num_classes * counts[i])
    weights.append(weight)

class_weights_tensor = torch.FloatTensor(weights).to(device)

print("Class counts:", class_counts)
print("Class weights:", weights)

In [ ]:
class_names = train_dataset.classes
num_classes = len(class_names)

print(class_names)
print("Num classes:", num_classes)

In [ ]:
class DenseLayer(nn.Module):
    def __init__(self, in_ch, growth_rate):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, growth_rate, 3, padding=1, bias=False)
        )

    def forward(self, x):
        out = self.block(x)
        return torch.cat([x, out], 1)


class DenseBlock(nn.Module):
    def __init__(self, in_ch, num_layers, growth_rate):
        super().__init__()
        layers = []
        channels = in_ch
        for _ in range(num_layers):
            layers.append(DenseLayer(channels, growth_rate))
            channels += growth_rate
        self.block = nn.Sequential(*layers)
        self.out_channels = channels

    def forward(self, x):
        return self.block(x)


class Transition(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.trans = nn.Sequential(
            nn.BatchNorm2d(in_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.AvgPool2d(2)
        )

    def forward(self, x):
        return self.trans(x)


class SimplifiedMicroDenseNet(nn.Module):
    def __init__(self, num_classes, growth_rate=12):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 24, 3, padding=1, bias=False),
            nn.BatchNorm2d(24),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.block1 = DenseBlock(24, 2, growth_rate)
        self.trans1 = Transition(self.block1.out_channels, 32)

        self.block2 = DenseBlock(32, 2, growth_rate)
        self.trans2 = Transition(self.block2.out_channels, 48)

        self.block3 = DenseBlock(48, 2, growth_rate)
        self.trans3 = Transition(self.block3.out_channels, 64)

        self.block4 = DenseBlock(64, 2, growth_rate)

        final_ch = self.block4.out_channels

        self.final = nn.Sequential(
            nn.BatchNorm2d(final_ch),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1,1))
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(final_ch, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)

        x = self.trans1(self.block1(x))
        x = self.trans2(self.block2(x))
        x = self.trans3(self.block3(x))
        x = self.block4(x)

        x = self.final(x)
        x = self.fc(x)
        return x


model = SimplifiedMicroDenseNet(num_classes)
model = model.to(device)

In [ ]:
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

outputs = model(images)
print(outputs.shape, labels.min().item(), labels.max().item())

In [ ]:
alpha = 0.6
temperature = 4.0

ce_loss = nn.CrossEntropyLoss(weight=class_weights_tensor)
kl_loss = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def train_model_with_early_stopping_distillation(
    model,
    teacher_model,
    optimizer,
    num_epochs=20,
    patience=5
):
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 30)

        # ================= TRAIN =================
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            # ----- Teacher Forward -----
            with torch.no_grad():
                teacher_logits = teacher_model(inputs)

            # ----- Student Forward -----
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # ----- Distillation Loss -----
            soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
            soft_student = F.log_softmax(outputs / temperature, dim=1)

            loss_ce = ce_loss(outputs, labels)
            loss_kd = kl_loss(soft_student, soft_teacher) * (temperature ** 2)

            loss = alpha * loss_ce + (1 - alpha) * loss_kd

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels)

        epoch_train_loss = running_loss / len(train_dataset)
        epoch_train_acc = running_corrects.double() / len(train_dataset)

        train_losses.append(epoch_train_loss)
        train_accs.append(epoch_train_acc.cpu())

        # ================= VALIDATION =================
        model.eval()
        running_loss = 0.0
        running_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)

                loss = ce_loss(outputs, labels)

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

        epoch_val_loss = running_loss / len(val_dataset)
        epoch_val_acc = running_corrects.double() / len(val_dataset)

        val_losses.append(epoch_val_loss)
        val_accs.append(epoch_val_acc.cpu())

        print(f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f}")
        print(f"Val   Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

        # ================= EARLY STOPPING =================
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_micro_densenet_distilled.pth")
        else:
            patience_counter += 1
            print(f"EarlyStopping counter: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

    return model, train_losses, val_losses, train_accs, val_accs

In [ ]:
model, train_losses, val_losses, train_accs, val_accs = \
    train_model_with_early_stopping(
        model,
        criterion,
        optimizer,
        num_epochs=20,
        patience=5
    )

In [ ]:
test_acc = evaluate_test_accuracy(model, test_loader, device)
print(f"Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
from collections import defaultdict

def per_class_accuracy(model, dataloader, class_names, device):
    model.eval()
    correct = defaultdict(int)
    total = defaultdict(int)

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            for label, pred in zip(labels, preds):
                total[label.item()] += 1
                if label == pred:
                    correct[label.item()] += 1

    for idx, name in enumerate(class_names):
        acc = correct[idx] / total[idx]
        print(f"{name}: {acc * 100:.2f}%")

In [ ]:
per_class_accuracy(model, test_loader, class_names, device)

In [ ]:
import os

PLOT_DIR = "/kaggle/working/plots"
os.makedirs(PLOT_DIR, exist_ok=True)
import matplotlib.pyplot as plt

def save_loss_plot(train_losses, val_losses):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOT_DIR}/loss_curve.png", dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
def save_accuracy_plot(train_accs, val_accs):
    plt.figure()
    plt.plot(train_accs, label="Train Accuracy")
    plt.plot(val_accs, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOT_DIR}/accuracy_curve.png", dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

def save_confusion_matrix(model, dataloader, class_names, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, annot=True, fmt="d",
        xticklabels=class_names,
        yticklabels=class_names,
        cmap="Blues"
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.savefig(f"{PLOT_DIR}/confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def save_roc_curves(model, dataloader, class_names, device):
    model.eval()
    y_true, y_scores = [], []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)

            y_true.extend(labels.numpy())
            y_scores.extend(torch.softmax(outputs, dim=1).cpu().numpy())

    y_true = np.array(y_true)
    y_scores = np.array(y_scores)

    y_true_bin = label_binarize(y_true, classes=range(len(class_names)))

    plt.figure(figsize=(8, 6))
    for i, class_name in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{class_name} (AUC={roc_auc:.2f})")

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Multi-class ROC Curve")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"{PLOT_DIR}/roc_curve.png", dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
save_loss_plot(train_losses, val_losses)
save_accuracy_plot(train_accs, val_accs)

save_confusion_matrix(
    model,
    test_loader,
    class_names,
    device
)

save_roc_curves(
    model,
    test_loader,
    class_names,
    device
)